In [ ]:
import os
import dotenv
import duckdb
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.ops import unary_union
from shapely.geometry import box
import logging
import gc 
import builtins 
from typing import List, Tuple, Dict, Optional
gc.enable()

gpd.options.io_engine = "pyogrio"
os.environ["PYOGRIO_USE_ARROW"] = "1"

dotenv.load_dotenv()
output_parquets=os.getenv('OUTPUT_PARQUETS')
input_parquet=os.getenv("INPUT_PARQUETS")
ret_class=os.path.join(input_parquet,'rec_class_all.parquet')
wrk_dir=os.getenv("WORKING_DIR")
brwmb_targets_csv=os.getenv("BRWMB_TARGETS")
source_data=os.getenv("source_data")
aflb_parquet=os.getenv("ALFB_PARQUET")
cia_dissolve=os.path.join(input_parquet,"cia_dissolve.parquet")
step2d_base=os.path.join(output_parquets,"step_2d_base_aflb.parquet")
table_outputs=os.getenv('TABLE_OUPUTS')
#set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
info=logging.info
debug=logging.debug
error=logging.error

#variables 
output_dir = os.path.join(wrk_dir, "brwmb_parquets")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
input_data_gdb=os.path.join(source_data,'Scen_5_inputs.gdb')
cia_ciap_l='cultural_areas'
hv1_l='hv1_clip'
tsu_l='TSU'

import duckdb
conn = duckdb.connect()
conn.install_extension("httpfs")
conn.install_extension("spatial")
conn.install_extension("arrow")
conn.load_extension("spatial")
conn.load_extension("httpfs")
conn.load_extension("arrow")

In [32]:
rec=gpd.read_parquet(ret_class)
rec['aflb_area_ha']=rec['aflb_fact']*(rec.geometry.area/10000)
for_rep_area=rec.groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat_short']).agg(total_aflb_ha=('aflb_area_ha', 'sum')).reset_index()
for_rep_area['desired total(25%) area(ha)']=for_rep_area['total_aflb_ha']*0.25

In [33]:
for_rep_area.to_excel(os.path.join(table_outputs,'for_rep_area.xlsx'), index=False)

In [34]:
wmb_aflb_from_rec=for_rep_area.groupby('WATER_MANAGEMENT_BASIN_NAME').agg(total_aflb_ha_from_rec=('total_aflb_ha', 'sum')).reset_index()

In [5]:
# wmb=gpd.read_file(input_data_gdb,layer='priority_WMB')
# wmb.rename(columns={'NAME':'WATER_MANAGEMENT_BASIN_NAME','SUM_aflb_ha':'WMB Total AFLB (ha)'},inplace=True)
# wmb[r'25% of WMB Total AFLB (ha)'] = wmb[r'WMB Total AFLB (ha)'] * 0.25
# wmb['total basin area(ha)'] = wmb.geometry.area/10000

In [6]:
# aflb_totals=wmb[['WATER_MANAGEMENT_BASIN_NAME','total basin area(ha)','WMB Total AFLB (ha)']].merge(wmb_aflb_from_rec,on='WATER_MANAGEMENT_BASIN_NAME',how='left')
# aflb_totals[r'% of WMB Total AFLB from Rec'] = (aflb_totals[r'total_aflb_ha_from_rec'] / aflb_totals[r'WMB Total AFLB (ha)']) * 100 
# aflb_totals.to_excel(os.path.join(table_outputs,'aflb_totals.xlsx'), index=False)

In [35]:
#create empty tables 

cols = ["HPC", "HPD", "HPM",
        "MPC", "MPD", "MPM",
        "LPC", "LPD", "LPM"]

index = [
    "Rec_Cat = 1",
    "Rec_Cat = 2",
    "Rec_Cat = 3",
    "Rec_Cat = 4",
    "Total AFLB (ha)"
]
empty_table = pd.DataFrame(0, index=index, columns=cols)

In [36]:
#build bluberry table 
bbr= rec[rec['WATER_MANAGEMENT_BASIN_NAME'] == 'Blueberry River']    
bbr_recs=bbr.groupby(['Rec_Cat','Rec_Cat_short']).agg(total_aflb_ha=('aflb_area_ha', 'sum')).reset_index()
bbr_recs_tbl=empty_table.copy().astype(float)
for index, row in bbr_recs.iterrows():
    rec_cat = row['Rec_Cat']
    rec_cat_short = row['Rec_Cat_short']
    total_aflb_ha = row['total_aflb_ha']
    
    if rec_cat == 1 and rec_cat_short in ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM']:
        bbr_recs_tbl.loc['Rec_Cat = 1', rec_cat_short] += total_aflb_ha
    elif rec_cat == 2 and rec_cat_short in ['MPD','LPD']:
        bbr_recs_tbl.loc['Rec_Cat = 2', rec_cat_short] += total_aflb_ha
bb_for_rep_area=for_rep_area[for_rep_area['WATER_MANAGEMENT_BASIN_NAME']=='Blueberry River'].copy()
bb_for_rep_area.rename(columns={'desired total(25%) area(ha)':'Total AFLB (ha)'}, inplace=True)
row_vals = (bb_for_rep_area
            .set_index('Rec_Cat_short')['Total AFLB (ha)'])
bbr_recs_tbl.loc['Total AFLB (ha)', cols] = row_vals.reindex(cols).values


In [37]:

cols = ["HPC","HPD","HPM","MPC","MPD","MPM","LPC","LPD","LPM"]
order = [1, 2, 3, 4]  # Rec_Cat priority

def allocate_from_source_to_target(
    bbr_recs, bbr_recs_tbl, cols,
    order=(1, 2, 3, 4),
    bottom_row='Total AFLB (ha)'
):
    """
    Use bbr_recs as the only source of area by (Rec_Cat, class),
    fill bbr_recs_tbl rows 'Rec_Cat = 1..4' per column in priority order,
    and ensure the column total never exceeds the bottom-row target.
    Works even if bbr_recs_tbl already has some prefilled values:
      - trims prefilled values to <= available supply and <= remaining target
      - then tops up from remaining supply until target is met (or supply exhausted)
    """

    # Ensure numeric dtype on target columns
    bbr_recs_tbl[list(cols)] = bbr_recs_tbl[list(cols)].astype(float)

    # Build a dict: (rec_cat:int, class:str) -> available_ha (float)
    src_series = (
        bbr_recs.copy()
                 .assign(Rec_Cat=lambda d: d['Rec_Cat'].astype(int))
                 .groupby(['Rec_Cat', 'Rec_Cat_short'])['total_aflb_ha']
                 .sum()
    )
    src = { (rc, cls): float(val) for (rc, cls), val in src_series.items() }

    # Row labels in priority order
    row_labels = [f"Rec_Cat = {rc}" for rc in order]

    # Process each class column independently
    for c in cols:
        # ----- read target -----
        try:
            target = float(bbr_recs_tbl.at[bottom_row, c])
        except KeyError:
            raise KeyError(
                f"Bottom row '{bottom_row}' or column '{c}' not found "
                f"in bbr_recs_tbl.index/columns"
            )

        remaining = target

        # ----- PASS 1: normalize any prefilled values -----
        # Trim prefilled cells so they do not exceed supply or target.
        for rc in order:
            row_label = f"Rec_Cat = {rc}"
            cur = float(bbr_recs_tbl.at[row_label, c])  # what is already in the table
            avail = float(src.get((rc, c), 0.0))        # supply for this rc/class

            if remaining <= 0.0:
                # Target already met; zero out anything prefilled in later rows
                if cur != 0.0:
                    bbr_recs_tbl.at[row_label, c] = 0.0
                continue

            # How much of the prefilled value can we keep?
            # It cannot exceed remaining target nor available supply.
            keep = builtins.min(cur, avail, remaining)

            if keep != cur:
                # Reduce prefilled amount to the allowable amount
                bbr_recs_tbl.at[row_label, c] = keep

            # Consume target and supply by the kept amount
            remaining -= keep
            avail -= keep

            # Update remaining supply for this rc/class (for PASS 2)
            src[(rc, c)] = avail

        # If target is met by prefilled amounts alone, move on
        if remaining <= 0.0:
            # Ensure any later rows are zero (already handled above)
            continue

        # ----- PASS 2: add from remaining supply in order until hitting target -----
        for rc in order:
            if remaining <= 0.0:
                break
            row_label = f"Rec_Cat = {rc}"
            avail = float(src.get((rc, c), 0.0))
            if avail <= 0.0:
                continue

            add = avail if avail <= remaining else remaining
            bbr_recs_tbl.at[row_label, c] = float(bbr_recs_tbl.at[row_label, c]) + add
            remaining -= add
            src[(rc, c)] = avail - add  # reduce supply

        # After PASS 2, remaining may be > 0 (not enough supply); that's fine.
        # Crucially, we never exceed target.

    return bbr_recs_tbl


In [38]:
bbr_recs_tbl = allocate_from_source_to_target(bbr_recs, bbr_recs_tbl, cols)

bbr_recs_tbl.to_excel(os.path.join(table_outputs,'bbr_recs_allocated.xlsx'))
bbr_recs.to_excel(os.path.join(table_outputs,'bbr_recs.xlsx'), index=False)

In [39]:
#build Lower Sikanni Chief River table 
bbr= rec[rec['WATER_MANAGEMENT_BASIN_NAME'] == 'Lower Sikanni Chief River']    
bbr_recs=bbr.groupby(['Rec_Cat','Rec_Cat_short']).agg(total_aflb_ha=('aflb_area_ha', 'sum')).reset_index()
bbr_recs_tbl=empty_table.copy().astype(float)
for index, row in bbr_recs.iterrows():
    rec_cat = row['Rec_Cat']
    rec_cat_short = row['Rec_Cat_short']
    total_aflb_ha = row['total_aflb_ha']
    
    if rec_cat == 1 and rec_cat_short in ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM']:
        bbr_recs_tbl.loc['Rec_Cat = 1', rec_cat_short] += total_aflb_ha
    elif rec_cat == 2 and rec_cat_short in ['HPC','HPM','MPD','LPC','LPD','LPM']:
        bbr_recs_tbl.loc['Rec_Cat = 2', rec_cat_short] += total_aflb_ha
    elif rec_cat == 3 and rec_cat_short in ['HPC']:
        bbr_recs_tbl.loc['Rec_Cat = 3', rec_cat_short] += total_aflb_ha
bb_for_rep_area=for_rep_area[for_rep_area['WATER_MANAGEMENT_BASIN_NAME']=='Lower Sikanni Chief River'].copy()
bb_for_rep_area.rename(columns={'desired total(25%) area(ha)':'Total AFLB (ha)'}, inplace=True)
row_vals = (bb_for_rep_area
            .set_index('Rec_Cat_short')['Total AFLB (ha)'])
bbr_recs_tbl.loc['Total AFLB (ha)', cols] = row_vals.reindex(cols).values

bbr_recs_tbl = allocate_from_source_to_target(bbr_recs, bbr_recs_tbl, cols)
bbr_recs_tbl.to_excel(os.path.join(table_outputs,'lsc_recs_allocated.xlsx'))
bbr_recs.to_excel(os.path.join(table_outputs,'lsc_recs.xlsx'), index=False)

In [40]:
#build 'Middle Beatton River' table 
bbr= rec[rec['WATER_MANAGEMENT_BASIN_NAME'] == 'Middle Beatton River']    
bbr_recs=bbr.groupby(['Rec_Cat','Rec_Cat_short']).agg(total_aflb_ha=('aflb_area_ha', 'sum')).reset_index()
bbr_recs_tbl=empty_table.copy().astype(float)
for index, row in bbr_recs.iterrows():
    rec_cat = row['Rec_Cat']
    rec_cat_short = row['Rec_Cat_short']
    total_aflb_ha = row['total_aflb_ha']
    
    if rec_cat == 1 and rec_cat_short in ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM']:
        bbr_recs_tbl.loc['Rec_Cat = 1', rec_cat_short] += total_aflb_ha
    elif rec_cat == 2 and rec_cat_short in ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM']:
        bbr_recs_tbl.loc['Rec_Cat = 2', rec_cat_short] += total_aflb_ha
    elif rec_cat == 3 and rec_cat_short in ['HPC','HPD','HPM']:
        bbr_recs_tbl.loc['Rec_Cat = 3', rec_cat_short] += total_aflb_ha
bb_for_rep_area=for_rep_area[for_rep_area['WATER_MANAGEMENT_BASIN_NAME']=='Middle Beatton River'].copy()
bb_for_rep_area.rename(columns={'desired total(25%) area(ha)':'Total AFLB (ha)'}, inplace=True)
row_vals = (bb_for_rep_area
            .set_index('Rec_Cat_short')['Total AFLB (ha)'])
bbr_recs_tbl.loc['Total AFLB (ha)', cols] = row_vals.reindex(cols).values

bbr_recs_tbl = allocate_from_source_to_target(bbr_recs, bbr_recs_tbl, cols)
bbr_recs_tbl.to_excel(os.path.join(table_outputs,'mbr_recs_allocated.xlsx'))
bbr_recs.to_excel(os.path.join(table_outputs,'mbr_recs.xlsx'), index=False)

In [41]:
#build Upper Beatton River table 
bbr= rec[rec['WATER_MANAGEMENT_BASIN_NAME'] == 'Upper Beatton River']    
bbr_recs=bbr.groupby(['Rec_Cat','Rec_Cat_short']).agg(total_aflb_ha=('aflb_area_ha', 'sum')).reset_index()
bbr_recs_tbl=empty_table.copy().astype(float)
for index, row in bbr_recs.iterrows():
    rec_cat = row['Rec_Cat']
    rec_cat_short = row['Rec_Cat_short']
    total_aflb_ha = row['total_aflb_ha']
    
    if rec_cat == 1 and rec_cat_short in ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM']:
        bbr_recs_tbl.loc['Rec_Cat = 1', rec_cat_short] += total_aflb_ha
    elif rec_cat == 2 and rec_cat_short in ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM']:
        bbr_recs_tbl.loc['Rec_Cat = 2', rec_cat_short] += total_aflb_ha
    elif rec_cat == 3 and rec_cat_short in ['HPC','HPD','HPM']:
        bbr_recs_tbl.loc['Rec_Cat = 3', rec_cat_short] += total_aflb_ha
    elif rec_cat == 4 and rec_cat_short in ['HPC','HPM']:
        bbr_recs_tbl.loc['Rec_Cat = 4', rec_cat_short] += total_aflb_ha
bb_for_rep_area=for_rep_area[for_rep_area['WATER_MANAGEMENT_BASIN_NAME']=='Upper Beatton River'].copy()
bb_for_rep_area.rename(columns={'desired total(25%) area(ha)':'Total AFLB (ha)'}, inplace=True)
row_vals = (bb_for_rep_area
            .set_index('Rec_Cat_short')['Total AFLB (ha)'])
bbr_recs_tbl.loc['Total AFLB (ha)', cols] = row_vals.reindex(cols).values

bbr_recs_tbl = allocate_from_source_to_target(bbr_recs, bbr_recs_tbl, cols)
bbr_recs_tbl.to_excel(os.path.join(table_outputs,'ubr_recs_allocated.xlsx'))
bbr_recs.to_excel(os.path.join(table_outputs,'ubr_recs.xlsx'), index=False)